In [28]:
import pandas as pd

import numpy as np
import xarray as xr

import os
from pathlib import Path

In [30]:
from utils.event_iden_funcs import (
    build_events_df
)


In [31]:
def select_pc1_events(lv=10, basepath="./processed", thresh=1.0, persistence=None,
    data_source: str | None = "era5", 
    latitude: str | None = "-20_to_-90",
    minsep=60):
    """
    Detect daily EOF1-based events from standardized PC1 time series.

    Parameters
    ----------
    lv : int
        Pressure level (default 50 hPa).
    basepath : str
        Directory containing daily.pc1.z{lv}.{YYYY}{MM}.txt files.
    thresh : float
        Threshold for event detection (default 1.0).
    persistence : int or None
        Number of consecutive days PC1 must stay above threshold.
        If None: defaults are 10 (for 50 hPa) or 14 (for 10/1 hPa).
    minsep : int
        Minimum days between independent events (default 60).
    outfile : str or None
        Path to save events. If None, auto-generate.

    Returns
    -------
    events : list of (year, month, day)
        Detected independent events.
    counts : dict
        Number of events per year.
    """

    # --- persistence defaults by level ---
    if persistence is None:
        if lv == 50:
            persistence = 10
        elif lv in (10, 1):
            persistence = 14
        else:
            raise ValueError(f"No default persistence set for level {lv}. Please provide persistence manually.")

    years = np.arange(1979, 2024)
    eday = [30,31,31,30,31,30,31]  # days in June–Dec
    cumdays = np.cumsum([0]+eday)
    tot_day = cumdays[-1]

    # --- load daily PC1 into array (year, day) ---
    data = np.full((len(years), tot_day), np.nan)
    for yi, yr in enumerate(years):
        for mi, nd in enumerate(eday):
            mon = mi+6
            basepath = Path(basepath)
            zeof_dir = basepath / "zeof"
            fname = os.path.join(zeof_dir, f"daily.pc1.z{lv}.{yr}{mon:02d}.txt")
            arr = np.loadtxt(fname, ndmin=2)
            vals = arr[0,1:nd+1] if arr.ndim==2 else arr[1:nd+1]
            data[yi, cumdays[mi]:cumdays[mi+1]] = vals

    events = []
    counts = {}

    # --- detection loop ---
    for yi, yr in enumerate(years):
        candidates = []
        for d in range(tot_day - persistence):
            # require day d and next `persistence` days > thresh
            if np.all(data[yi, d:d+persistence+1] > thresh):
                candidates.append(d)

        kept = []
        if candidates:
            lastd = candidates[0]
            kept.append(lastd)
            for dd in candidates[1:]:
                if dd - lastd >= minsep:
                    kept.append(dd)
                    lastd = dd

        # convert to (year, month, day)
        for dd in kept:
            mon_idx = np.searchsorted(cumdays, dd, side='right')-1
            mon = mon_idx+6
            day_in_mon = dd - cumdays[mon_idx] + 1
            events.append(pd.Timestamp(year=yr, month=mon, day=day_in_mon))

        counts[yr] = len(kept)

    event_dates = pd.to_datetime(events)

    # --- build events DataFrame ---
    min_gap_days = minsep
    
    # --- build standardized events DataFrame ---
    events_df = build_events_df(
        dates=event_dates,
        method="pc1",
        definition=f"EOF1_PC1_gt{thresh:.1f}_{int(lv)}hPa_{latitude}",
        data_source=data_source or "",
        threshold=f">{thresh:.1f}σ",
        level_hpa=str(lv),
        latitude=str(latitude),
        notes=(
            f"PC1 > {thresh:.1f}σ for at least {persistence} consecutive days; "
            f"events separated by ≥{min_gap_days} days"
        ),
        # extra_cols={"n_events": len(event_dates)}  # optional if build_events_df supports it
    )

    return events_df, event_dates

In [32]:
events_df, event_dates = select_pc1_events(lv=10)

In [33]:
events_df

,date,method,definition,data_source,threshold,level_hpa,latitude,lat_band,notes
0,1979-09-27,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
1,1979-12-09,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
2,1980-11-07,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
3,1981-09-17,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
4,1981-11-27,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
5,1982-11-11,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
6,1988-08-08,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
7,1988-10-07,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
8,1989-12-06,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...
9,1991-10-05,pc1,EOF1_PC1_gt1.0_10hPa_-20_to_-90,era5,>1.0σ,10,-20_to_-90,<NA>,PC1 > 1.0σ for at least 14 consecutive days; e...


- adapt

In [2]:
def detect_ssw_pc1_events(
    pc1_std_all: np.ndarray | xr.DataArray,
    years: np.ndarray,
    *,
    lv: int = 10,
    thresh: float = 1.0,
    persistence: int | None = None,
    min_gap_days: int = 60,
    months: list[int] | None = None,
    data_source: str | None = "era5",
    latitude: str | None = "-20_to_-90",
    output_csv: bool | str = False,
) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    """
    Detect EOF1-based SSW-like events from standardized daily PC1 data.

    Parameters
    ----------
    pc1_std_all : np.ndarray or xr.DataArray
        Array of standardized daily PC1 (shape: [n_years, n_days]).
        Typically output from `compute_pc1_all_months()`.
    years : array-like
        Year labels corresponding to the first axis of pc1_std_all.
    lv : int, optional
        Pressure level (default: 10 hPa).
    thresh : float, optional
        Threshold for event detection (default: 1.0).
    persistence : int or None, optional
        Minimum number of consecutive days above threshold.
        Defaults: 10 for 50 hPa, 14 for 10/1 hPa.
    min_gap_days : int, optional
        Minimum separation (days) between independent events (default: 60).
    months : list[int], optional
        List of months covered (default: June–December).
    data_source : str, optional
        Data source label (default: "era5").
    latitude : str, optional
        Latitude range (for metadata).
    output_csv : bool or str, optional
        If True, write to current folder; if str, write to given path.

    Returns
    -------
    events_df : pd.DataFrame
        Detected event information with metadata.
    event_dates : pd.DatetimeIndex
        Array of detected event dates.
    """

    # --- set defaults ---
    if persistence is None:
        if lv == 50:
            persistence = 10
        elif lv in (10, 1):
            persistence = 14
        else:
            raise ValueError(f"No default persistence for level {lv}. Provide manually.")
    months = months or list(range(6, 13))  # June–December default

    # --- setup day indexing per month (for event localization) ---
    days_in_mon = [30, 31, 31, 30, 31, 30, 31]  # June–Dec
    cumdays = np.cumsum([0] + days_in_mon)
    total_days = cumdays[-1]

    if pc1_std_all.shape[1] != total_days:
        raise ValueError(
            f"pc1_std_all has {pc1_std_all.shape[1]} days, expected {total_days} for June–Dec"
        )

    events, counts = [], {}

    # --- detection loop ---
    for yi, yr in enumerate(years):
        candidates = []
        arr = pc1_std_all[yi, :]

        for d in range(total_days - persistence):
            if np.all(arr[d : d + persistence + 1] > thresh):
                candidates.append(d)

        # remove events closer than min_gap_days
        kept = []
        if candidates:
            last = candidates[0]
            kept.append(last)
            for dd in candidates[1:]:
                if dd - last >= min_gap_days:
                    kept.append(dd)
                    last = dd

        # convert indices to dates
        for dd in kept:
            mon_idx = np.searchsorted(cumdays, dd, side="right") - 1
            mon = months[mon_idx]
            day_in_mon = dd - cumdays[mon_idx] + 1
            events.append(pd.Timestamp(year=int(yr), month=int(mon), day=int(day_in_mon)))

        counts[int(yr)] = len(kept)

    event_dates = pd.to_datetime(events)

    # --- build events DataFrame ---
    events_df = pd.DataFrame(
        dict(date=event_dates),
    )
    events_df["method"] = "pc1"
    events_df["definition"] = f"EOF1_PC1_gt{thresh:.1f}_{lv}hPa"
    events_df["data_source"] = data_source
    events_df["threshold"] = thresh
    events_df["level_hpa"] = lv
    events_df["latitude"] = latitude
    events_df["persist_days"] = persistence
    events_df["min_gap_days"] = min_gap_days
    events_df["n_events"] = len(events_df)

    # --- optional save ---
    if output_csv:
        out_dir = Path(".") if output_csv is True else Path(output_csv)
        out_dir.mkdir(parents=True, exist_ok=True)

        outfile = out_dir / f"EOF1_PC1_gt{thresh:.1f}_{lv}hPa_events.csv"
        with open(outfile, "w") as fout:
            fout.write("# EOF1-based daily PC1 events\n")
            fout.write(f"# PC1 > {thresh} for ≥{persistence} days, min gap {min_gap_days} days\n")
            fout.write(f"# Level: {lv} hPa, Source: {data_source}, Lat: {latitude}\n")
            events_df[["date"]].to_csv(fout, index=False)
        print(f" Saved {len(event_dates)} events → {outfile}")

    return events_df, event_dates


In [ ]:

events_df, event_dates = detect_ssw_pc1_events(
    pc1_std_all=pc1_all,
    years=years,
    lv=10,
    thresh=1.0,
    persistence=14,
    min_gap_days=60,
    output_csv="./processed",
)